In [1]:
# IMPORT FILES FROM DATASET
import os
from json.decoder import JSONArray

from jupyter_server.utils import fetch

files = [file for file in os.listdir('data/') if file.endswith(".pgn")]

In [2]:
len(files)

1

In [3]:
from chess import pgn

def load_pgn(file):
    games = []
    with open(file, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)

    return games

In [4]:
from tqdm import tqdm

games = []

for file in tqdm(files):
    games.extend(load_pgn('data/' + file))

100%|██████████| 1/1 [00:19<00:00, 19.80s/it]


In [5]:
print(games[0].board().outcome())

None


In [6]:
len(games)

8728

In [7]:
# IMPORTS FOR NEURAL NETWORK
import numpy as np
from chess import Board
from chess import PAWN, KNIGHT, BISHOP, ROOK, KING, QUEEN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Input, BatchNormalization, MaxPooling2D, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.regularizers import l2

2025-04-30 21:52:38.348915: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746021158.397962   61478 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746021158.412159   61478 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746021158.495848   61478 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746021158.495881   61478 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746021158.495883   61478 computation_placer.cc:177] computation placer alr

In [8]:
# turn the board into a matrix
def board_to_matrix(board : Board):
    matrix = np.zeros((8, 8, 16))
    piece_map = board.piece_map()

    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        piece_eval = 1 if piece.color else -1

        if piece.piece_type == PAWN:
            piece_eval *= 10
        elif piece.piece_type == KNIGHT:
            piece_eval *= 30
        elif piece.piece_type == BISHOP:
            piece_eval *= 30
        elif piece.piece_type == ROOK:
            piece_eval *= 50
        elif piece.piece_type == QUEEN:
            piece_eval *= 90
        elif piece.piece_type == KING:
            piece_eval *= 900

        matrix[row, col, piece_type + piece_color] = piece_eval

    legal_moves = board.legal_moves
    pseudo_moves = board.pseudo_legal_moves

    for move in legal_moves:
        to_square = move.to_square
        row_to, col_to = divmod(to_square, 8)
        matrix[row_to, col_to, 12] = 1

        if board.piece_at(move.to_square):
            matrix[row_to, col_to, 13] = 1

    for move in pseudo_moves:
        to_square = move.to_square
        row_to, col_to = divmod(to_square, 8)

        if board.gives_check(move):
            matrix[row_to, col_to, 15] = 1
        else:
            matrix[row_to, col_to, 14] = 1

    return matrix

# inputs for possible moves
def input_for_nn(games):
    X = []
    y = []
    for game in games:
        print(f'game {games.index(game) + 1} / {len(games)}')
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return X, y

# convert moves
def encode_moves(moves):
    print("encoding...")
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int


In [9]:
X, y = input_for_nn(games[:2000])
y, move_to_int = encode_moves(y)
print(f'length of move_to_int:{len(move_to_int)}')
y = to_categorical(y, num_classes=len(move_to_int))
print(f'length of y:{len(y)}')

game 1 / 2000
game 2 / 2000
game 3 / 2000
game 4 / 2000
game 5 / 2000
game 6 / 2000
game 7 / 2000
game 8 / 2000
game 9 / 2000
game 10 / 2000
game 11 / 2000
game 12 / 2000
game 13 / 2000
game 14 / 2000
game 15 / 2000
game 16 / 2000
game 17 / 2000
game 18 / 2000
game 19 / 2000
game 20 / 2000
game 21 / 2000
game 22 / 2000
game 23 / 2000
game 24 / 2000
game 25 / 2000
game 26 / 2000
game 27 / 2000
game 28 / 2000
game 29 / 2000
game 30 / 2000
game 31 / 2000
game 32 / 2000
game 33 / 2000
game 34 / 2000
game 35 / 2000
game 36 / 2000
game 37 / 2000
game 38 / 2000
game 39 / 2000
game 40 / 2000
game 41 / 2000
game 42 / 2000
game 43 / 2000
game 44 / 2000
game 45 / 2000
game 46 / 2000
game 47 / 2000
game 48 / 2000
game 49 / 2000
game 50 / 2000
game 51 / 2000
game 52 / 2000
game 53 / 2000
game 54 / 2000
game 55 / 2000
game 56 / 2000
game 57 / 2000
game 58 / 2000
game 59 / 2000
game 60 / 2000
game 61 / 2000
game 62 / 2000
game 63 / 2000
game 64 / 2000
game 65 / 2000
game 66 / 2000
game 67 / 2000
game

In [11]:
print(f'length of X:{len(X)}')
X = np.array(X)
print(f'length of X:{len(X)}')

length of X:188808
length of X:188808


In [12]:

model = Sequential([
    Input(shape=(8, 8, 16)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Flatten(),
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(len(move_to_int), activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

I0000 00:00:1746021231.544321   61478 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2274 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 8, 8, 64)       │         9,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1826)           │       235,554 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,368,162 (5.22 MB)

 Trainable params: 1,367,778 (5.22 MB)

 Non-trainable params: 384 (1.50 KB)

In [13]:
history = model.fit(X, y, epochs=75, validation_split=0.1, batch_size=128)
model.save('hikarubot_2.keras')

2025-04-30 21:53:56.077200: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 696020992 exceeds 10% of free system memory.
2025-04-30 21:53:58.138153: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1241146808 exceeds 10% of free system memory.
2025-04-30 21:53:59.703464: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 696020992 exceeds 10% of free system memory.
2025-04-30 21:54:00.079082: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1241146808 exceeds 10% of free system memory.


Epoch 1/75


I0000 00:00:1746021242.610525   61714 service.cc:152] XLA service 0x775ccc005880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1746021242.610550   61714 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Ti Laptop GPU, Compute Capability 8.6
2025-04-30 21:54:02.682476: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1746021242.976738   61714 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-04-30 21:54:03.947373: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 56 bytes spill stores, 56 bytes spill loads

2025-04-30 21:54:04.124796: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fu

  32/1328 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.0050 - loss: 7.7827

I0000 00:00:1746021248.650560   61714 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1327/1328 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0179 - loss: 6.8890

2025-04-30 21:54:15.997057: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 452 bytes spill stores, 356 bytes spill loads

2025-04-30 21:54:16.046684: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 36 bytes spill stores, 36 bytes spill loads

2025-04-30 21:54:16.152265: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135_0', 120 bytes spill stores, 120 bytes spill loads

2025-04-30 21:54:16.393648: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 60 bytes spill stores, 60 bytes spill loads

2025-04-30 21:54:16.497685: I exte

1328/1328 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0179 - loss: 6.8888

2025-04-30 21:54:22.006403: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_122', 88 bytes spill stores, 88 bytes spill loads

2025-04-30 21:54:22.478910: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 152.00MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-04-30 21:54:24.506598: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_122', 108 bytes spill stores, 108 bytes spill loads

2025-04-30 21:54:24.537725: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_122', 48 bytes 

1328/1328 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.0179 - loss: 6.8885 - val_accuracy: 0.0573 - val_loss: 6.0025
Epoch 2/75
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.0461 - loss: 6.1288 - val_accuracy: 0.0842 - val_loss: 5.7106
Epoch 3/75
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.0710 - loss: 5.8517 - val_accuracy: 0.1077 - val_loss: 5.4001
Epoch 4/75
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.0945 - loss: 5.5226 - val_accuracy: 0.1410 - val_loss: 5.0436
Epoch 5/75
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.1200 - loss: 5.1955 - val_accuracy: 0.1616 - val_loss: 4.7587
Epoch 6/75
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.1410 - loss: 4.9124 - val_accuracy: 0.1705 - val_loss: 4.5619
Epoch 7/75
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.1568 - loss: 4.7016 - val_accuracy: 0.1881 - val_loss: 4.4053
Epoch 8/75
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.1719 - loss: 4.5222 - val_accur

In [31]:
# Load model
from tensorflow.keras.models import load_model
model = load_model('./hikarubot_24_3.keras')

In [45]:
int_to_move = dict(zip(move_to_int.values(), move_to_int.keys()))

def predict_move(board : Board):
    board_matrix = board_to_matrix(board).reshape(1, 8, 8, 15)
    prediction = model.predict(board_matrix)
    move = int_to_move[np.argmax(prediction)]
    print(np.argmax(prediction))
    return move

In [53]:

board = Board()


In [55]:
# while not board.is_game_over():
#     board.push_uci(input("Your move:"))

next_move = predict_move(board)
board.push_uci(next_move)

print(f"Predicted move: {next_move}")
print(board)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1491
Predicted move: c7c5
r n b q k b n r
p p . p p p p p
. . . . . . . .
. . p . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R


In [16]:
array = np.array(move_to_int)
np.save('move_to_int_24_3.npy', move_to_int)